In [ ]:
## Loading data, definitions, and cohort metadata; adjusting coordinates

# ----------------------------------------------------------------------
# Imports and data locations
# ----------------------------------------------------------------------
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

DATA_DIR = "data2019"

# ----------------------------------------------------------------------
# Load the experiment definitions
# ----------------------------------------------------------------------
# definitions_2019 contains experiment-specific information such as
# the experiment start date and camera geometry.
sys.path.append(DATA_DIR)
import definitions_2019 as bd

# ----------------------------------------------------------------------
# Load the trajectory data for the selected day
# ----------------------------------------------------------------------
DAY = 4
traj_file = os.path.join(DATA_DIR, f"beetrajectories_{DAY:03d}.hdf")
df = pd.read_hdf(traj_file)

# Print the experiment date corresponding to the selected day.
print("Experiment starts on:",bd.startday)
current_date = bd.startday + pd.Timedelta(days=DAY)
print("Current Date:",current_date)

# ----------------------------------------------------------------------
# Add bee age and cohort information
# ----------------------------------------------------------------------
# daydatamat.csv contains the mapping from bee UID to age/cohort for
# each experimental day. Restrict it to the selected day first.
cohort_df = pd.read_csv("daydatamat.csv")
cohort_df=cohort_df[cohort_df["Day number"]==DAY]

uid_to_age = dict(zip(cohort_df["Bee unique ID"], cohort_df["Age"]))
df["age"] = df["uid"].map(uid_to_age)

uid_to_cohort = dict(zip(cohort_df["Bee unique ID"], cohort_df["Cohort ID"]))
df["cohort"] = df["uid"].map(uid_to_cohort)

# ----------------------------------------------------------------------
# Correct the x-coordinate for camera 0
# ----------------------------------------------------------------------
df.loc[df["camera"] == 0, "x"] += bd.xpixels


In [ ]:
## Finding out the framegaps for each bee and filling in the MAX_GAP frames linearly

import numpy as np
import pandas as pd
import gc

MAX_GAP = 5

# ======================================================================
# Gap statistics
# ======================================================================
# Find missing frames separately for each bee. A gap of length N means
# that N frames are missing between two consecutive observations.

gap_lengths = []

for _, grp in df.groupby("uid", sort=False):
    frames = grp["framenum"].to_numpy()
    gaps = np.diff(frames) - 1
    gap_lengths.extend(gaps[gaps > 0])

gap_lengths = np.asarray(gap_lengths)

# Count the occurrence of each gap length and calculate its fraction
# and cumulative fraction among all detected gaps.
gap_counts = (
    pd.Series(gap_lengths)
      .value_counts()
      .sort_index()
      .rename_axis("Gap length")
      .reset_index(name="Count")
)

gap_counts["Fraction"] = gap_counts["Count"] / gap_counts["Count"].sum()
gap_counts["Cumulative"] = gap_counts["Fraction"].cumsum()

print(gap_counts)

print("\nSummary:")
for t in [1, 2, 3, 5]:
    frac = gap_counts.loc[gap_counts["Gap length"] <= t, "Fraction"].sum()
    print(f"Gap ≤ {t:2d}: {100*frac:6.2f}% of all gaps")

# ======================================================================
# Interpolate gaps up to MAX_GAP frames — vectorised
# ======================================================================
# Only short gaps are filled. Longer gaps are left untouched because
# interpolation across them would be less reliable.

df = df.sort_values(["uid", "framenum"]).reset_index(drop=True)

# Convert the required columns to NumPy arrays for vectorised processing.
# Each index refers to one observation in df.
uid_arr    = df["uid"].to_numpy()
frame_arr  = df["framenum"].to_numpy()
x_arr      = df["x"].to_numpy(dtype=float)
y_arr      = df["y"].to_numpy(dtype=float)
theta_arr  = df["theta"].to_numpy(dtype=float)

# Identify consecutive observations belonging to the same bee and
# having a gap between 1 and MAX_GAP missing frames.
same_uid   = uid_arr[:-1] == uid_arr[1:]
gap_arr    = frame_arr[1:] - frame_arr[:-1] - 1   # gap between row i and i+1
fill_mask  = same_uid & (gap_arr >= 1) & (gap_arr <= MAX_GAP)
fill_idx   = np.where(fill_mask)[0]   # indices i where we need to interpolate

print(f"\nGaps to fill: {len(fill_idx):,}")

if len(fill_idx) > 0:
    # --------------------------------------------------------------
    # Expand each gap into the individual interpolated rows
    # --------------------------------------------------------------
    # gap_sizes[k] is the number of missing rows for the k-th gap.
    gap_sizes  = gap_arr[fill_idx]
    total_rows = gap_sizes.sum()

    # Repeat the starting observation once for every row that must
    # be inserted into its corresponding gap.
    rep_idx    = np.repeat(fill_idx, gap_sizes)

    # j_within gives the position of each inserted frame inside its gap:
    # 1, 2, ..., gap_size.
    j_within   = np.ones(total_rows, dtype=int)
    pos = 0
    for k, gs in enumerate(gap_sizes):
        j_within[pos:pos+gs] = np.arange(1, gs+1)
        pos += gs

    # Faster: build j_within without Python loop using repeat + cumsum trick
    # (replace the loop above)
    counts     = gap_sizes
    j_within   = np.arange(total_rows) - np.repeat(
        np.concatenate(([0], np.cumsum(counts[:-1]))), counts
    ) + 1

    # Fractional position of each interpolated point between its two
    # observed endpoints.
    frac       = j_within / (gap_arr[rep_idx] + 1)

    # --------------------------------------------------------------
    # Interpolate position and orientation
    # --------------------------------------------------------------
    # Angular interpolation uses the shortest angular path so that
    # values crossing the -pi/pi boundary are handled correctly.
    dtheta     = (theta_arr[rep_idx + 1] - theta_arr[rep_idx] + np.pi) % (2*np.pi) - np.pi

    interp_frame = frame_arr[rep_idx] + j_within
    interp_x     = x_arr[rep_idx]     + frac * (x_arr[rep_idx + 1]     - x_arr[rep_idx])
    interp_y     = y_arr[rep_idx]     + frac * (y_arr[rep_idx + 1]     - y_arr[rep_idx])
    interp_theta = theta_arr[rep_idx] + frac * dtheta
    interp_theta = (interp_theta + np.pi) % (2*np.pi) - np.pi

    # --------------------------------------------------------------
    # Build the interpolated dataframe
    # --------------------------------------------------------------
    # All columns other than framenum/x/y/theta are inherited from
    # the observation at the start of the corresponding gap.
    interp_df = df.iloc[rep_idx].copy().reset_index(drop=True)
    interp_df["framenum"] = interp_frame
    interp_df["x"]        = interp_x
    interp_df["y"]        = interp_y
    interp_df["theta"]    = interp_theta

    print(f"Adding {len(interp_df):,} interpolated rows...")

    # Append the interpolated observations and restore chronological
    # ordering within each bee.
    df = pd.concat([df, interp_df], ignore_index=True)
    df.sort_values(["uid", "framenum"], inplace=True)
    df.reset_index(drop=True, inplace=True)

print(f"Final dataframe size: {len(df):,} rows")

# ======================================================================
# Cleanup
# ======================================================================
# These intermediate objects are no longer needed after interpolation.
del gap_lengths, gap_counts
gc.collect()


In [ ]:
## Loading the comb image for the selected experimental day

import displayfunctions as bp
import pickle
import gzip

# The comb image is stored as a compressed pickle and is selected using
# the same DAY value used for the trajectory data.
zfilln = 3
comb_contents_dir = 'comb-contents-images2019/'
comb = pickle.load(gzip.open(comb_contents_dir+'comb_'+str(DAY).zfill(zfilln)+'.pklz','rb'))


## Festoon Scoring Scheme & Pipeline Architecture

This section calculates a composite **Festoon Score** for each bee over short time windows. Because festooning bees form static, vertically oriented, linked chains, the scoring algorithm heavily penalizes movement and horizontal orientations. 

**Note: A lower overall score indicates a higher likelihood of festooning.**

### 1. The 5 Core Metrics
Each metric is calculated per bee, per frame-chunk, and is min-max normalized to a `[0, 1]` scale:
*   **Speed (30% weight):** Average translational speed. Festooning bees are largely immobile, so slower bees score better.
*   **Angular Speed (20% weight):** Average rate of rotation. Festooning bees maintain a locked orientation, so lower angular speed scores better.
*   **Positional Stability (20% weight):** The sum of the standard deviations of the bee's X and Y coordinates. This measures spatial "wiggling" or drifting. 
*   **Verticality (15% weight):** The angular deviation from a perfectly vertical orientation (±π/2). 
*   **Chain Membership (15% weight):** The fraction of frames the bee spends physically close to, and pointed toward, at least two other vertically aligned bees (computed via `cKDTree`). Because a *higher* fraction is better, this specific score is inverted (`1 - chain_fraction`) so that 0 remains the best possible score.

### 2. How the Score is Used Downstream
1.  **Candidate Selection per Frame-Chunk:** For every frame-chunk (e.g., a 30-frame window), the bees are sorted by their `festoon_score`. The top *N* bees (e.g., the 30 with the lowest scores) are temporarily saved as primary festoon candidates for that specific chunk.
2.  **Tracking Persistence (Chunk Appearances):** We then aggregate these results across the entire observation window to count how many times each unique bee appeared in the top *N* candidates for a chunk. This metric, known as "chunk appearances," effectively measures a bee's structural persistence. The bees that appear in the highest number of chunks are identified as our top overall festooners.
3.  **Establishing a Global Threshold:** By isolating a 6-hour "reference window" and ranking the bees by their chunk appearances, we identify the 100 most persistent festooners. The worst (highest) single score ever recorded by the 100th bee in this group becomes our definitive, data-driven **`threshold_score`**.
4.  **24-Hour Population Sweep:** We wrap this scoring pipeline into a function and run it across the entire 24-hour dataset. Any bee that achieves a score below the `threshold_score` in a given time block is officially classified as a "festooner."
5.  **Demographic Analysis:** With our classified festooners, we map the results back to the cohort metadata to analyze the age distribution of the festoon, calculate how long individual bees sustain continuous festooning bouts (streaks), and observe how the festoon size changes hour-by-hour.

In [ ]:
## Main festoon scoring pipeline: filtering, calculating speeds, detecting vertical chains, and aggregating scores per chunk

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

# Work on a copy so that the original trajectory dataframe remains available.
df2 = df.copy()
#df2.loc[df2["camera"] == 0, "x"] += bd.xpixels

# ======================================================================
# Analysis parameters
# ======================================================================
# BOX_X / BOX_Y define the spatial region analysed.
# FRAME_START / FRAME_END define the reference time window.
# The remaining parameters define the criteria used to identify
# festooning bees and vertical chains.
BOX_X          = (100, 6500)
BOX_Y          = (100, 1820)
FRAME_START    = 32398*2*3
FRAME_END      = 32398*2*4
MIN_OBS        = 20          # bee must appear in at least this many frames
MAX_SPEED      = 15.0        # px/s — festooners are nearly static
VERT_TOL       = np.pi / 4   # theta within 45° of vertical (±π/2)
CHAIN_DIST     = 250         # px — max distance to a vertical neighbor
CHAIN_ANGLE    = np.pi / 4   # how closely bee must point toward neighbor
MIN_CHAIN_SIZE = 3           # chain must have at least this many bees
CHUNK_SIZE = 30*3   # frames per chunk (10 seconds at 3 FPS)

# ======================================================================
# Restrict the data to the analysis box and reference time window
# ======================================================================
df_box = df2[
    (df2["framenum"].between(FRAME_START, FRAME_END)) &
    (df2["x"].between(*BOX_X)) &
    (df2["y"].between(*BOX_Y))
].copy()

# ======================================================================
# Calculate linear and angular speed
# ======================================================================
# Differences are calculated separately for each bee. dframe is retained
# so that speed remains correct when observations are separated by more
# than one frame.
df_box = df_box.sort_values(["uid", "framenum"])

df_box["dx"]     = df_box.groupby("uid")["x"].diff()
df_box["dy"]     = df_box.groupby("uid")["y"].diff()
df_box["dframe"] = df_box.groupby("uid")["framenum"].diff()

# Angular displacement with wrap-around correction.
df_box["dtheta_raw"] = df_box.groupby("uid")["theta"].diff()
df_box["dtheta"]     = np.arctan2(
    np.sin(df_box["dtheta_raw"]),
    np.cos(df_box["dtheta_raw"])
)

FRAMES_PER_SECOND = 3
N_MAX             = 10   # tolerate up to N_MAX frame gaps

# Only use observations separated by a valid number of frames.
valid = (df_box["dframe"] >= 1) & (df_box["dframe"] <= N_MAX)

df_box["time_elapsed"]  = df_box["dframe"] / FRAMES_PER_SECOND
df_box["displacement"]  = np.sqrt(df_box["dx"]**2 + df_box["dy"]**2)
df_box["speed"]         = np.where(valid, df_box["displacement"] / df_box["time_elapsed"], np.nan)
df_box["angular_speed"] = np.where(valid, np.abs(df_box["dtheta"]) / df_box["time_elapsed"], np.nan)

# Clip extreme values before aggregation so isolated tracking errors do not
# dominate the per-chunk averages.
speed_999    = df_box["speed"].quantile(0.999)
angspd_999   = df_box["angular_speed"].quantile(0.999)
df_box["speed"]         = df_box["speed"].clip(upper=speed_999)
df_box["angular_speed"] = df_box["angular_speed"].clip(upper=angspd_999)

print(f"Speed 99.9th pct: {speed_999:.1f} px/s")
print(f"Angular speed 99.9th pct: {angspd_999:.3f} rad/s")

# ======================================================================
# Divide the reference window into chunks
# ======================================================================
df_box["chunk"] = (df_box["framenum"] - FRAME_START) // CHUNK_SIZE

# ======================================================================
# Determine vertical-chain membership for every frame
# ======================================================================
print("Computing per-frame vertical chains...")
frame_chain_sets = {}
chain_counts     = {}

def bees_in_vertical_chains(group):
    uids  = group["uid"].values
    xy    = group[["x", "y"]].values
    theta = group["theta"].values
    n     = len(uids)
    if n < 2:
        return set()

    # Find spatially close bee pairs efficiently.
    tree = cKDTree(xy)
    adj  = {i: set() for i in range(n)}

    # Build an adjacency graph using both distance and orientation criteria.
    for i, j in tree.query_pairs(CHAIN_DIST):
        dx, dy = xy[j] - xy[i]

        # A vertical neighbour must have a larger vertical than horizontal
        # separation.
        if abs(dx) > abs(dy):
            continue

        angle_to_j = np.arctan2(dy, dx)
        angle_to_i = np.arctan2(-dy, -dx)

        diff_i = abs(np.arctan2(np.sin(theta[i] - angle_to_j),
                                np.cos(theta[i] - angle_to_j)))
        diff_j = abs(np.arctan2(np.sin(theta[j] - angle_to_i),
                                np.cos(theta[j] - angle_to_i)))

        # A pair qualifies if at least one bee points sufficiently toward
        # the other bee.
        if diff_i < CHAIN_ANGLE or diff_j < CHAIN_ANGLE:
            adj[i].add(j)
            adj[j].add(i)

    # Find connected components in the adjacency graph. Components with
    # at least MIN_CHAIN_SIZE bees are considered qualifying chains.
    visited, chain_uids = set(), set()
    for start in range(n):
        if start in visited or start not in adj or not adj[start]:
            continue

        component, queue = [], [start]
        while queue:
            node = queue.pop()
            if node in visited:
                continue
            visited.add(node)
            component.append(node)
            queue.extend(adj[node] - visited)

        if len(component) >= MIN_CHAIN_SIZE:
            chain_uids.update(uids[k] for k in component)

    return chain_uids

# Store the chain members for each frame and count how often each bee
# participates in a chain.
for framenum, group in df_box.groupby("framenum"):
    in_chain = bees_in_vertical_chains(group)
    frame_chain_sets[framenum] = in_chain
    for uid in in_chain:
        chain_counts[uid] = chain_counts.get(uid, 0) + 1

# Convert frame-level chain membership into a row-level indicator.
df_box["in_chain"] = df_box.apply(
    lambda r: float(r["uid"] in frame_chain_sets.get(r["framenum"], set())), axis=1
)

# ======================================================================
# Aggregate features for each bee in each chunk
# ======================================================================
print("Aggregating per chunk...")

per_bee_chunk = df_box.groupby(["uid", "chunk"]).agg(
    mean_speed     = ("speed",         "mean"),
    mean_ang_speed = ("angular_speed", "mean"),   # ← new
    n_obs          = ("framenum",      "count"),
    mean_x         = ("x",             "mean"),
    mean_y         = ("y",             "mean"),
    std_x          = ("x",             "std"),
    std_y          = ("y",             "std"),
    mean_theta     = ("theta",         "mean"),
    chain_fraction = ("in_chain",      "mean"),
    age            = ("age",           "first"),
    cohort         = ("cohort",        "first"),
    frame_start    = ("framenum",      "min"),
    frame_end      = ("framenum",      "max"),
).reset_index()

# Discard very sparsely observed bee-chunk combinations.
per_bee_chunk = per_bee_chunk[per_bee_chunk["n_obs"] >= 5].copy()

# ======================================================================
# Calculate the individual components of the festoon score
# ======================================================================
# Distance from either vertical orientation (+pi/2 or -pi/2).
# Smaller values mean the bee is more vertically oriented.
per_bee_chunk["vert_score"] = np.minimum(
    np.abs(np.arctan2(np.sin(per_bee_chunk["mean_theta"] - np.pi/2),
                      np.cos(per_bee_chunk["mean_theta"] - np.pi/2))),
    np.abs(np.arctan2(np.sin(per_bee_chunk["mean_theta"] + np.pi/2),
                      np.cos(per_bee_chunk["mean_theta"] + np.pi/2)))
)

# Positional variability: smaller values correspond to greater stability.
per_bee_chunk["pos_stability"] = (per_bee_chunk["std_x"].fillna(999) +
                                   per_bee_chunk["std_y"].fillna(999))

# Min-max normalization within each chunk.
def norm01(s):
    r = s.max() - s.min()
    return (s - s.min()) / r if r > 0 else s * 0

per_bee_chunk["score_speed"]     = per_bee_chunk.groupby("chunk")["mean_speed"].transform(norm01)
per_bee_chunk["score_ang_speed"] = per_bee_chunk.groupby("chunk")["mean_ang_speed"].transform(norm01)  # ← new
per_bee_chunk["score_stability"] = per_bee_chunk.groupby("chunk")["pos_stability"].transform(norm01)
per_bee_chunk["score_vert"]      = per_bee_chunk.groupby("chunk")["vert_score"].transform(norm01)

# Higher chain fraction indicates stronger chain participation, so invert it
# because lower values are intended to represent better festoon candidates.
per_bee_chunk["score_chain"]     = 1 - per_bee_chunk["chain_fraction"]

# ======================================================================
# Combine the five criteria into the final festoon score
# ======================================================================
# Lower festoon_score = stronger festoon candidate.
# The weights below sum to 1.
per_bee_chunk["festoon_score"] = (
    per_bee_chunk["score_speed"]     * 0.30 +
    per_bee_chunk["score_ang_speed"] * 0.20 +   # ← new
    per_bee_chunk["score_stability"] * 0.20 +
    per_bee_chunk["score_vert"]      * 0.15 +
    per_bee_chunk["score_chain"]     * 0.15
)

# ======================================================================
# Select the top N festoon candidates in each chunk
# ======================================================================
TOP_N = 30
top_per_chunk = (
    per_bee_chunk
    .sort_values("festoon_score")
    .groupby("chunk")
    .head(TOP_N)
    .reset_index(drop=True)
)

# Keep the top UIDs in score order. This lookup is used later by the
# animation worker for tier-based colouring/identification.
chunk_uid_map = {}
for chunk_idx, grp in top_per_chunk.groupby("chunk"):
    chunk_uid_map[chunk_idx] = (
        grp.sort_values("festoon_score")["uid"].tolist()
    )

print(f"Chunks computed: {per_bee_chunk['chunk'].nunique()}")
print(f"Chunk summary (top {TOP_N} per chunk):")
print(top_per_chunk.groupby("chunk")[["festoon_score", "mean_speed",
                                       "mean_ang_speed", "chain_fraction"]].mean().round(3))


In [ ]:
## Interactive view of festoon candidates in the reference window

%matplotlib qt
import matplotlib.pyplot as plt
import displayfunctions as bp
from matplotlib.patches import Rectangle
from matplotlib.widgets import Slider

# Create the figure and a slider that allows the user to move through
# individual frames in the reference window.
fig, ax = plt.subplots(figsize=(14, 10))
plt.subplots_adjust(bottom=0.20)

ax_slider = plt.axes([0.20, 0.05, 0.65, 0.03])
frame_slider = Slider(ax=ax_slider, label="Frame",
                      valmin=FRAME_START, valmax=FRAME_END,
                      valinit=FRAME_START, valstep=1)

def update(val):
    # --------------------------------------------------------------
    # Determine the selected frame and its corresponding chunk
    # --------------------------------------------------------------
    frame_val  = int(frame_slider.val)
    chunk_idx  = (frame_val - FRAME_START) // CHUNK_SIZE

    # Top UIDs for this chunk
    chunk_top  = top_per_chunk[top_per_chunk["chunk"] == chunk_idx]
    top_uids   = set(chunk_top["uid"].values)

    # --------------------------------------------------------------
    # Draw the comb, analysis box, and bees
    # --------------------------------------------------------------
    ax.clear()
    bp.showcomb(comb, ax=ax)
    ax.add_patch(Rectangle((BOX_X[0], BOX_Y[0]),
                            BOX_X[1]-BOX_X[0], BOX_Y[1]-BOX_Y[0],
                            linewidth=2, edgecolor="green", facecolor="none"))

    frame_df = df_box[df_box["framenum"] == frame_val]
    cand_df  = frame_df[frame_df["uid"].isin(top_uids)]
    other_df = frame_df[~frame_df["uid"].isin(top_uids)]

    # Non-candidates are shown faintly in the background.
    ax.scatter(other_df["x"], other_df["y"], s=12, color="grey", alpha=0.3)

    # Highlight candidate bees and show their heading with a quiver arrow.
    if not cand_df.empty:
        ax.scatter(cand_df["x"], cand_df["y"], s=80, color="red",
                   edgecolors="black", linewidths=0.8, zorder=4,
                   label=f"festoon candidates ({len(cand_df)} present)")
        ax.quiver(cand_df["x"].values, cand_df["y"].values,
                  np.cos(cand_df["theta"].values), np.sin(cand_df["theta"].values),
                  color="yellow", angles="xy", scale_units="xy",
                  scale=1/40, width=0.004, zorder=5)

    # Annotate each visible candidate with its festoon score in this chunk.
    score_map = dict(zip(chunk_top["uid"], chunk_top["festoon_score"]))
    for _, row in frame_df[frame_df["uid"].isin(top_uids)].iterrows():
        score = score_map.get(row["uid"], 0)
        ax.annotate(f"{score:.2f}", (row["x"], row["y"]),
                    fontsize=6, color="white", ha="center", zorder=6)

    # Display the current frame, chunk, and number of candidates present.
    ax.set_title(f"Frame {frame_val}  |  Chunk {chunk_idx}  "
                 f"(frames {FRAME_START + chunk_idx*CHUNK_SIZE}–"
                 f"{FRAME_START + (chunk_idx+1)*CHUNK_SIZE - 1})  |  "
                 f"{len(cand_df)}/{TOP_N} candidates present")
    ax.legend(loc="upper right")
    fig.canvas.draw_idle()

frame_slider.on_changed(update)
update(FRAME_START)
plt.show()


In [ ]:
## Identifying persistent festooners across chunks

# ======================================================================
# Count how often each bee appears among the top candidates
# ======================================================================
# A bee that repeatedly appears in the top candidates across chunks is
# treated as an actual festooner.
chunk_appearances = (
    top_per_chunk
    .groupby("uid")
    .agg(
        chunks_in_top = ("chunk", "count"),
        age           = ("age",   "first"),
        cohort        = ("cohort","first"),
        mean_x        = ("mean_x","mean"),
        mean_y        = ("mean_y","mean"),
    )
    .reset_index()
    .sort_values("chunks_in_top", ascending=False)
)

print(f"Bees that appeared in top {TOP_N} in at least one chunk: {len(chunk_appearances)}")
print(chunk_appearances.head(20)[["uid","age","chunks_in_top","mean_x","mean_y"]].to_string())

# ======================================================================
# Select the most persistent festooners
# ======================================================================
TOP_FESTOON = 100   # change to 50 if preferred

top_festooners = chunk_appearances.head(TOP_FESTOON).copy()
total_chunks   = per_bee_chunk["chunk"].nunique()
top_festooners["fraction_of_chunks"] = top_festooners["chunks_in_top"] / total_chunks

print(f"\nTop {TOP_FESTOON} festooners:")
print(top_festooners[["uid","age","chunks_in_top",
                       "fraction_of_chunks","mean_x","mean_y"]].to_string())

# ======================================================================
# Summarise the persistent festooners by age
# ======================================================================
age_counts = top_festooners.groupby("age")["uid"].count().reset_index()
age_counts.columns = ["age", "n_bees"]

# Mean number of chunks spent among the persistent festooners of each age.
age_persistence = top_festooners.groupby("age")["chunks_in_top"].mean().reset_index()
age_persistence.columns = ["age", "mean_chunks"]

# ======================================================================
# Plot age distribution and persistence
# ======================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: number of bees per age in top festooners
age_list_fest = sorted(top_festooners["age"].unique())
colors        = [plt.cm.tab10(i % 10) for i, _ in enumerate(age_list_fest)]
color_map     = dict(zip(age_list_fest, colors))

ax1.bar(age_counts["age"], age_counts["n_bees"],
        color=[color_map[a] for a in age_counts["age"]],
        edgecolor="black", linewidth=0.7)
ax1.set_xlabel("Age (days)")
ax1.set_ylabel("Number of bees")
ax1.set_title(f"Top {TOP_FESTOON} festooners — count by age")
ax1.set_xticks(age_counts["age"])
for _, row in age_counts.iterrows():
    ax1.text(row["age"], row["n_bees"] + 0.3, str(int(row["n_bees"])),
             ha="center", fontsize=9)
ax1.grid(axis="y", alpha=0.3)

# Right: mean chunks spent among the persistent festooners of each age.
ax2.bar(age_persistence["age"], age_persistence["mean_chunks"],
        color=[color_map[a] for a in age_persistence["age"]],
        edgecolor="black", linewidth=0.7)
ax2.set_xlabel("Age (days)")
ax2.set_ylabel(f"Mean chunks in top {TOP_N} (out of {total_chunks})")
ax2.set_title(f"Top {TOP_FESTOON} festooners — avg. time in festoon by age")
ax2.set_xticks(age_persistence["age"])
for _, row in age_persistence.iterrows():
    ax2.text(row["age"], row["mean_chunks"] + 0.1, f"{row['mean_chunks']:.1f}",
             ha="center", fontsize=9)
ax2.grid(axis="y", alpha=0.3)

plt.suptitle(
    f"Festoon age distribution  |  {total_chunks} chunks of {CHUNK_SIZE} frames  "
    f"|  top {TOP_N} per chunk  |  top {TOP_FESTOON} by persistence",
    fontsize=11
)
plt.tight_layout()
plt.savefig("festoon_age_histogram.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
## Comparing festooning contribution across bee age groups

%matplotlib qt

# ======================================================================
# Count all observed bees in each age group
# ======================================================================
bees_per_age = (
    df_box.groupby("age")["uid"]
    .nunique()
    .reset_index()
    .rename(columns={"uid": "total_bees"})
)

# ======================================================================
# Calculate age-specific festooner metrics
# ======================================================================
# 1. Fraction of each age group represented among the top persistent festooners.
age_counts = top_festooners.groupby("age")["uid"].count().reset_index()
age_counts.columns = ["age", "n_festooners"]
age_counts = age_counts.merge(bees_per_age, on="age")
age_counts["fraction_festooning"] = age_counts["n_festooners"] / age_counts["total_bees"]

# 2. Mean number of chunks spent in the top candidates, among the
# persistent festooners only.
age_presence = top_festooners.groupby("age")["chunks_in_top"].mean().reset_index()
age_presence.columns = ["age", "mean_chunks"]

# 3. Total additive chunk appearances across all persistent festooners
# in each age group.
age_total_time = top_festooners.groupby("age")["chunks_in_top"].sum().reset_index()
age_total_time.columns = ["age", "total_chunks"]

# 4. Normalize the total contribution by the total number of bees observed
# in that age group, giving an approximate per-capita contribution.
age_counts = age_counts.merge(age_total_time, on="age")
age_counts["normalized_contribution"] = age_counts["total_chunks"] / age_counts["total_bees"]

print(age_counts[["age", "n_festooners", "total_bees", "fraction_festooning", "normalized_contribution"]].to_string())

# ======================================================================
# Plot the four age-based measures
# ======================================================================
age_list_fest = sorted(age_counts["age"].unique())
colors        = [plt.cm.tab10(i % 10) for i, _ in enumerate(age_list_fest)]
color_map     = dict(zip(age_list_fest, colors))

fig, (ax1, ax2, ax3, ax4) = plt.subplots(1, 4, figsize=(30, 6))

# Panel 1: Fraction of each age group represented among persistent festooners.
ax1.bar(age_counts["age"], age_counts["fraction_festooning"],
        color=[color_map[a] for a in age_counts["age"]],
        edgecolor="black", linewidth=0.7)
ax1.set_xlabel("Age (days)")
ax1.set_ylabel("Fraction of age group in top festooners")
ax1.set_title(f"Fraction of each age group festooning\n(top {TOP_FESTOON} by chunk presence)")
ax1.set_xticks(age_counts["age"])
for _, row in age_counts.iterrows():
    ax1.text(row["age"], row["fraction_festooning"] + 0.002,
             f"{row['fraction_festooning']:.2f}\n({int(row['n_festooners'])}/{int(row['total_bees'])})",
             ha="center", fontsize=8)
ax1.grid(axis="y", alpha=0.3)

# Panel 2: Mean persistence among the top festooners.
ax2.bar(age_presence["age"], age_presence["mean_chunks"],
        color=[color_map[a] for a in age_presence["age"]],
        edgecolor="black", linewidth=0.7)
ax2.set_xlabel("Age (days)")
ax2.set_ylabel(f"Mean chunks in top {TOP_N}")
ax2.set_title(f"Avg. festoon presence by age\n(among top {TOP_FESTOON} festooners only)")
ax2.set_xticks(age_presence["age"])
for _, row in age_presence.iterrows():
    ax2.text(row["age"], row["mean_chunks"] + (age_presence["mean_chunks"].max() * 0.02),
             f"{row['mean_chunks']:.1f}",
             ha="center", fontsize=9)
ax2.grid(axis="y", alpha=0.3)

# Panel 3: Total additive bee-chunk contribution.
ax3.bar(age_total_time["age"], age_total_time["total_chunks"],
        color=[color_map[a] for a in age_total_time["age"]],
        edgecolor="black", linewidth=0.7)
ax3.set_xlabel("Age (days)")
ax3.set_ylabel("Total chunk appearances (additive)")
ax3.set_title(f"Total festoon contribution by age\n(absolute bee-chunks)")
ax3.set_xticks(age_total_time["age"])
for _, row in age_total_time.iterrows():
    ax3.text(row["age"], row["total_chunks"] + (age_total_time["total_chunks"].max() * 0.02),
             f"{int(row['total_chunks'])}",
             ha="center", fontsize=9)
ax3.grid(axis="y", alpha=0.3)

# Panel 4: Total contribution normalized by the number of bees in the age group.
ax4.bar(age_counts["age"], age_counts["normalized_contribution"],
        color=[color_map[a] for a in age_counts["age"]],
        edgecolor="black", linewidth=0.7)
ax4.set_xlabel("Age (days)")
ax4.set_ylabel(f"Expected chunks per bee (out of {total_chunks})")
ax4.set_title(f"Normalized contribution by age\n(per capita bee-chunks)")
ax4.set_xticks(age_counts["age"])
for _, row in age_counts.iterrows():
    ax4.text(row["age"], row["normalized_contribution"] + (age_counts["normalized_contribution"].max() * 0.02),
             f"{row['normalized_contribution']:.2f}",
             ha="center", fontsize=9)
ax4.grid(axis="y", alpha=0.3)

plt.suptitle(
    f"Festoon age demographics  |  {total_chunks} chunks of {CHUNK_SIZE} frames  "
    f"|  top {TOP_N} per chunk  |  top {TOP_FESTOON} by presence",
    fontsize=14
)

# pad=2.0 forces a margin around the entire outer edge of the figure so labels aren't clipped.
# w_pad=4.0 keeps the horizontal breathing room between the individual panels.
# rect=[0, 0, 1, 0.95] explicitly reserves the top 5% of the figure strictly for the suptitle.
plt.tight_layout(pad=4.0, w_pad=4.0, rect=[0, 0, 1, 0.95]) 

plt.savefig("festoon_age_histogram_four_panels.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
## Measuring continuous festooning bouts among the persistent festooners

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import MaxNLocator

# ======================================================================
# Restrict the analysis to the top persistent festooners
# ======================================================================
top_100_uids = top_festooners["uid"].values
streak_df = top_per_chunk[top_per_chunk["uid"].isin(top_100_uids)].copy()

# ======================================================================
# Sort observations chronologically for each bee
# ======================================================================
streak_df = streak_df.sort_values(["uid", "chunk"])

# ======================================================================
# Identify continuous sequences of occupied chunks
# ======================================================================
# A continuous sequence of chunks will maintain a constant difference
# when a sequential rank is subtracted from the chunk number.
streak_df["chunk_rank"] = streak_df.groupby("uid")["chunk"].rank(method="first")
streak_df["streak_group"] = streak_df["chunk"] - streak_df["chunk_rank"]

# ======================================================================
# Calculate the length of every continuous streak
# ======================================================================
all_streaks = (
    streak_df.groupby(["uid", "streak_group"])
    .size()
    .reset_index(name="streak_len")
)

# ======================================================================
# Find the longest continuous streak for each bee
# ======================================================================
max_streaks_per_bee = all_streaks.groupby("uid")["streak_len"].max().reset_index()

# ======================================================================
# Plot streak distributions
# ======================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: longest continuous streak for each bee.
# The -0.5 offset centres histogram bins on integer chunk counts.
bins_max = np.arange(0, max_streaks_per_bee["streak_len"].max() + 2) - 0.5

ax1.hist(max_streaks_per_bee["streak_len"], bins=bins_max, 
         color="mediumseagreen", edgecolor="black", linewidth=1.2)
ax1.set_xlabel("Max Continuous Chunks")
ax1.set_ylabel("Number of Bees")
ax1.set_title(f"Longest Continuous Bout per Bee\n(Top {TOP_FESTOON} Festooners)")
ax1.xaxis.set_major_locator(MaxNLocator(integer=True))
ax1.grid(axis="y", linestyle="--", alpha=0.7)

# Panel 2: distribution of every individual continuous streak.
bins_all = np.arange(0, all_streaks["streak_len"].max() + 2) - 0.5

ax2.hist(all_streaks["streak_len"], bins=bins_all, 
         color="cornflowerblue", edgecolor="black", linewidth=1.2)
ax2.set_xlabel("Continuous Chunks")
ax2.set_ylabel("Number of Bouts (Streaks)")
ax2.set_title(f"Distribution of All Festooning Bouts\n(Every streak by the top {TOP_FESTOON} bees)")
ax2.xaxis.set_major_locator(MaxNLocator(integer=True))
ax2.grid(axis="y", linestyle="--", alpha=0.7)

# Add summary statistics to the second panel.
mean_bout = all_streaks["streak_len"].mean()
total_bouts = len(all_streaks)
textstr = f"Total separate bouts: {total_bouts}\nMean bout length: {mean_bout:.2f} chunks"
props = dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='gray')
ax2.text(0.95, 0.95, textstr, transform=ax2.transAxes, fontsize=10,
         verticalalignment='top', horizontalalignment='right', bbox=props)

plt.suptitle(
    f"Festoon Structural Persistence  |  Chunk Size: {CHUNK_SIZE} frames",
    fontsize=14, y=1.05
)

plt.tight_layout(w_pad=4.0)
plt.savefig("festoon_persistence_streaks.png", dpi=150, bbox_inches="tight")
plt.show()

# ======================================================================
# Print summary statistics
# ======================================================================
print(f"Total unique bees evaluated: {len(max_streaks_per_bee)}")
print(f"Absolute longest single continuous bout: {all_streaks['streak_len'].max()} chunks")
print(f"Average continuous bout length: {all_streaks['streak_len'].mean():.2f} chunks")


In [ ]:
## Comparing continuous festooning bout length across ages

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ======================================================================
# Assign the age of each persistent festooner to every identified bout
# ======================================================================
# top_festooners contains the UID-to-age mapping for the bees being analysed.
age_dict = top_festooners.set_index("uid")["age"].to_dict()
all_streaks["age"] = all_streaks["uid"].map(age_dict)

# ======================================================================
# Calculate mean bout length and its uncertainty by age
# ======================================================================
bout_summary_by_age = (
    all_streaks.groupby("age")
    .agg(
        mean_bout_len=("streak_len", "mean"),
        std_bout_len=("streak_len", "std"),      # To calculate error bars
        total_bouts=("streak_len", "count")      # Number of separate shifts
    )
    .reset_index()
)

# Calculate Standard Error of the Mean (SEM) for the error bars.
# If an age group only has 1 bout, std is NaN, so it is replaced with 0.
bout_summary_by_age["sem_bout_len"] = (
    bout_summary_by_age["std_bout_len"] / np.sqrt(bout_summary_by_age["total_bouts"])
).fillna(0)

# ======================================================================
# Plot average continuous bout length by age
# ======================================================================
fig, ax = plt.subplots(figsize=(10, 6))

# Re-use the colour mapping convention from the previous age-based plots.
age_list_fest = sorted(bout_summary_by_age["age"].unique())
colors        = [plt.cm.tab10(i % 10) for i, _ in enumerate(age_list_fest)]
color_map     = dict(zip(age_list_fest, colors))

# Draw the mean bout length with SEM error bars.
ax.bar(
    bout_summary_by_age["age"], 
    bout_summary_by_age["mean_bout_len"],
    yerr=bout_summary_by_age["sem_bout_len"], 
    capsize=5,
    color=[color_map[a] for a in bout_summary_by_age["age"]],
    edgecolor="black", 
    linewidth=1.2,
    alpha=0.8
)

# Formatting
ax.set_xlabel("Age (days)", fontsize=12)
ax.set_ylabel(f"Avg Continuous Bout Length (Chunks of {CHUNK_SIZE} frames)", fontsize=12)
ax.set_title(f"Average Festooning Shift Duration by Age\n(Top {TOP_FESTOON} Festooners)", fontsize=14, pad=15)
ax.set_xticks(bout_summary_by_age["age"])

# Annotate the exact mean value slightly above the error bar.
for _, row in bout_summary_by_age.iterrows():
    y_pos = row["mean_bout_len"] + row["sem_bout_len"] + (bout_summary_by_age["mean_bout_len"].max() * 0.03)
    ax.text(row["age"], y_pos, f"{row['mean_bout_len']:.1f}", ha="center", fontsize=9)

ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("festoon_avg_bout_by_age.png", dpi=150, bbox_inches="tight")
plt.show()

# Print the underlying summary table for reference.
print(bout_summary_by_age[["age", "total_bouts", "mean_bout_len", "sem_bout_len"]].round(2).to_string())


In [ ]:
## Establishing the festoon-score threshold from the reference window

# ----------------------------------------------------------------------
# Select the top persistent festooners in the reference window
# ----------------------------------------------------------------------
# top_per_chunk and per_bee_chunk were calculated above for FRAME_START
# through FRAME_END. Here we rank bees by how many chunks they appeared in
# among the top candidates.
TOP_FESTOON = 100

chunk_appearances_ref = (
    top_per_chunk
    .groupby("uid")
    .agg(chunks_in_top=("chunk", "count"), age=("age", "first"))
    .reset_index()
    .sort_values("chunks_in_top", ascending=False)
)

# The 100th-ranked bee defines the reference threshold.
top100_ref   = chunk_appearances_ref.head(TOP_FESTOON)
bottom_uid   = top100_ref.iloc[-1]["uid"]

# The threshold is based on this bee's mean festoon score across the
# chunks in which they appeared. Lower scores are better.
threshold_score = per_bee_chunk[
    per_bee_chunk["uid"] == bottom_uid
]["festoon_score"].mean()

print(f"100th bee uid: {bottom_uid}")
print(f"Festoon score threshold: {threshold_score:.4f}")
print(f"(Any bee with festoon_score <= {threshold_score:.4f} in a chunk is called festooning)")


In [ ]:
## Running the festoon-scoring pipeline across the full day

# ======================================================================
# Define the hourly analysis windows
# ======================================================================
# One hour = 32398*2 / 6 frames  (since 32398*2 = 6 hours)
FRAMES_PER_SECOND = 3
FRAMES_PER_HOUR   = (32398 * 2) // 6     # ≈ 10799 frames

# Full day range
DAY_START = 0
DAY_END   = 32398 * 2 * 4 - 100

hour_starts = list(range(DAY_START, DAY_END, FRAMES_PER_HOUR))
print(f"Total hours to analyse: {len(hour_starts)}")
print(f"Frames per hour: {FRAMES_PER_HOUR}")

# ======================================================================
# Define the reusable hourly festoon pipeline
# ======================================================================
def run_festoon_pipeline(df_full, bd, hour_start, hour_end,
                         BOX_X, BOX_Y, CHUNK_SIZE,
                         CHAIN_DIST, CHAIN_ANGLE, MIN_CHAIN_SIZE,
                         N_MAX=10, TOP_N=30):
    """
    Run the full festoon scoring pipeline on a given frame window.
    Returns per_bee_chunk dataframe and top_per_chunk dataframe.
    """

    # ------------------------------------------------------------------
    # Restrict the data to the requested time window and analysis box
    # ------------------------------------------------------------------
    df2 = df_full.copy()
    df2.loc[df2["camera"] == 0, "x"] += bd.xpixels

    df_box = df2[
        (df2["framenum"].between(hour_start, hour_end)) &
        (df2["x"].between(*BOX_X)) &
        (df2["y"].between(*BOX_Y))
    ].copy()

    if df_box.empty:
        return None, None

    # ------------------------------------------------------------------
    # Calculate linear and angular speed
    # ------------------------------------------------------------------
    df_box = df_box.sort_values(["uid", "framenum"])
    df_box["dx"]         = df_box.groupby("uid")["x"].diff()
    df_box["dy"]         = df_box.groupby("uid")["y"].diff()
    df_box["dframe"]     = df_box.groupby("uid")["framenum"].diff()
    df_box["dtheta_raw"] = df_box.groupby("uid")["theta"].diff()
    df_box["dtheta"]     = np.arctan2(np.sin(df_box["dtheta_raw"]),
                                       np.cos(df_box["dtheta_raw"]))

    valid = (df_box["dframe"] >= 1) & (df_box["dframe"] <= N_MAX)
    df_box["time_elapsed"]  = df_box["dframe"] / FRAMES_PER_SECOND
    df_box["displacement"]  = np.sqrt(df_box["dx"]**2 + df_box["dy"]**2)
    df_box["speed"]         = np.where(valid, df_box["displacement"] / df_box["time_elapsed"], np.nan)
    df_box["angular_speed"] = np.where(valid, np.abs(df_box["dtheta"]) / df_box["time_elapsed"], np.nan)

    # Clip extreme values at the 99.9th percentile before aggregation.
    speed_999  = df_box["speed"].quantile(0.999)
    angspd_999 = df_box["angular_speed"].quantile(0.999)
    df_box["speed"]         = df_box["speed"].clip(upper=speed_999)
    df_box["angular_speed"] = df_box["angular_speed"].clip(upper=angspd_999)

    # ------------------------------------------------------------------
    # Assign each observation to a time chunk
    # ------------------------------------------------------------------
    df_box["chunk"] = (df_box["framenum"] - hour_start) // CHUNK_SIZE

    # ------------------------------------------------------------------
    # Determine vertical-chain membership for each frame
    # ------------------------------------------------------------------
    def bees_in_vertical_chains(group):
        uids  = group["uid"].values
        xy    = group[["x", "y"]].values
        theta = group["theta"].values
        n     = len(uids)
        if n < 2:
            return set()

        tree = cKDTree(xy)
        adj  = {i: set() for i in range(n)}

        # Build an adjacency graph from spatial proximity and orientation.
        for i, j in tree.query_pairs(CHAIN_DIST):
            dx, dy = xy[j] - xy[i]
            if abs(dx) > abs(dy):
                continue

            angle_to_j = np.arctan2(dy, dx)
            angle_to_i = np.arctan2(-dy, -dx)
            diff_i = abs(np.arctan2(np.sin(theta[i] - angle_to_j),
                                    np.cos(theta[i] - angle_to_j)))
            diff_j = abs(np.arctan2(np.sin(theta[j] - angle_to_i),
                                    np.cos(theta[j] - angle_to_i)))

            if diff_i < CHAIN_ANGLE or diff_j < CHAIN_ANGLE:
                adj[i].add(j)
                adj[j].add(i)

        # Find connected components. A component qualifies as a chain
        # if it contains at least MIN_CHAIN_SIZE bees.
        visited, chain_uids = set(), set()
        for start in range(n):
            if start in visited or start not in adj or not adj[start]:
                continue

            component, queue = [], [start]
            while queue:
                node = queue.pop()
                if node in visited:
                    continue
                visited.add(node)
                component.append(node)
                queue.extend(adj[node] - visited)

            if len(component) >= MIN_CHAIN_SIZE:
                chain_uids.update(uids[k] for k in component)

        return chain_uids

    # Apply the chain detector independently to every frame.
    in_chain_col = []
    for framenum, group in df_box.groupby("framenum"):
        chain_set = bees_in_vertical_chains(group)
        in_chain_col.extend(
            float(uid in chain_set) for uid in group["uid"].values
        )
    df_box = df_box.sort_values(["framenum", "uid"])
    df_box["in_chain"] = in_chain_col

    # ------------------------------------------------------------------
    # Aggregate per (uid, chunk)
    # ------------------------------------------------------------------
    per_bee_chunk = df_box.groupby(["uid", "chunk"]).agg(
        mean_speed     = ("speed",         "mean"),
        mean_ang_speed = ("angular_speed", "mean"),
        n_obs          = ("framenum",      "count"),
        mean_x         = ("x",             "mean"),
        mean_y         = ("y",             "mean"),
        std_x          = ("x",             "std"),
        std_y          = ("y",             "std"),
        mean_theta     = ("theta",         "mean"),
        chain_fraction = ("in_chain",      "mean"),
        age            = ("age",           "first"),
        cohort         = ("cohort",        "first"),
    ).reset_index()

    # Ignore bee-chunk combinations with fewer than 5 observations.
    per_bee_chunk = per_bee_chunk[per_bee_chunk["n_obs"] >= 5].copy()
    if per_bee_chunk.empty:
        return None, None

    # ------------------------------------------------------------------
    # Calculate the individual feature scores
    # ------------------------------------------------------------------
    per_bee_chunk["vert_score"] = np.minimum(
        np.abs(np.arctan2(np.sin(per_bee_chunk["mean_theta"] - np.pi/2),
                          np.cos(per_bee_chunk["mean_theta"] - np.pi/2))),
        np.abs(np.arctan2(np.sin(per_bee_chunk["mean_theta"] + np.pi/2),
                          np.cos(per_bee_chunk["mean_theta"] + np.pi/2)))
    )
    per_bee_chunk["pos_stability"] = (per_bee_chunk["std_x"].fillna(999) +
                                       per_bee_chunk["std_y"].fillna(999))

    def norm01(s):
        r = s.max() - s.min()
        return (s - s.min()) / r if r > 0 else s * 0

    per_bee_chunk["score_speed"]     = per_bee_chunk.groupby("chunk")["mean_speed"].transform(norm01)
    per_bee_chunk["score_ang_speed"] = per_bee_chunk.groupby("chunk")["mean_ang_speed"].transform(norm01)
    per_bee_chunk["score_stability"] = per_bee_chunk.groupby("chunk")["pos_stability"].transform(norm01)
    per_bee_chunk["score_vert"]      = per_bee_chunk.groupby("chunk")["vert_score"].transform(norm01)
    per_bee_chunk["score_chain"]     = 1 - per_bee_chunk["chain_fraction"]

    # ------------------------------------------------------------------
    # Combine the five criteria into the final festoon score
    # ------------------------------------------------------------------
    per_bee_chunk["festoon_score"] = (
        per_bee_chunk["score_speed"]     * 0.30 +
        per_bee_chunk["score_ang_speed"] * 0.20 +
        per_bee_chunk["score_stability"] * 0.20 +
        per_bee_chunk["score_vert"]      * 0.15 +
        per_bee_chunk["score_chain"]     * 0.15
    )

    # ------------------------------------------------------------------
    # Select the top N candidates in each chunk
    # ------------------------------------------------------------------
    top_per_chunk = (
        per_bee_chunk
        .sort_values("festoon_score")
        .groupby("chunk")
        .head(TOP_N)
        .reset_index(drop=True)
    )

    return per_bee_chunk, top_per_chunk


In [ ]:
## Checking the number of unique bees present in the analysis box per hour

# This is a diagnostic cell used to verify how many unique bees are
# available in the spatial region during each hourly analysis window.
for hour_idx, h_start in enumerate(hour_starts):
    h_end = h_start + FRAMES_PER_HOUR - 1
    n_bees = df2[
        (df2["framenum"].between(h_start, h_end)) &
        (df2["x"].between(*BOX_X)) &
        (df2["y"].between(*BOX_Y))
    ]["uid"].nunique()
    print(f"Hour {hour_idx:02d}: {n_bees} unique bees in box")


In [ ]:
## Checking the number of unique bees in each age group

# Simple diagnostic: count distinct bee UIDs for every age group.
print(df.groupby("age")["uid"].nunique())


In [ ]:
## Running the festoon pipeline for every hour and collecting metrics

from scipy.spatial import cKDTree

# ======================================================================
# Set up result containers
# ======================================================================
age_groups  = sorted(df["age"].dropna().unique())
results     = []
results_by_age = {a: [] for a in age_groups}

# ======================================================================
# Analyse each hourly window
# ======================================================================
for hour_idx, h_start in enumerate(hour_starts):
    h_end = h_start + FRAMES_PER_HOUR - 1
    print(f"Hour {hour_idx:02d}: frames {h_start}–{h_end}", end="  ", flush=True)

    # Run the same scoring pipeline used for the reference window.
    pbc, tpc = run_festoon_pipeline(
        df, bd,
        hour_start     = h_start,
        hour_end       = h_end,
        BOX_X          = BOX_X,
        BOX_Y          = BOX_Y,
        CHUNK_SIZE     = CHUNK_SIZE,
        CHAIN_DIST     = CHAIN_DIST,
        CHAIN_ANGLE    = CHAIN_ANGLE,
        MIN_CHAIN_SIZE = MIN_CHAIN_SIZE,
    )

    # ==================================================================
    # Overall metrics
    # ==================================================================
    if pbc is None:
        print("→ no data")
        results.append({
            "hour": hour_idx, "frame_start": h_start,
            "n_above_threshold": 0, "mean_top100_score": np.nan,
        })
        for age in age_groups:
            results_by_age[age].append({
                "hour": hour_idx,
                "n_above_threshold": 0, "mean_top100_score": np.nan,
            })
        continue

    # For each bee, use its best (minimum) festoon score across chunks.
    # Count how many bees meet the reference threshold.
    best_per_bee = pbc.groupby("uid")["festoon_score"].min().reset_index()
    n_above      = int((best_per_bee["festoon_score"] <= threshold_score).sum())

    # Identify the most persistent top candidates in this hour and calculate
    # their mean festoon score.
    chunk_app = (
        tpc.groupby("uid").agg(chunks_in_top=("chunk", "count"))
           .reset_index().sort_values("chunks_in_top", ascending=False)
    )
    top100_hour       = chunk_app.head(TOP_FESTOON)["uid"].values
    mean_top100_score = (
        pbc[pbc["uid"].isin(top100_hour)]["festoon_score"].mean()
        if len(top100_hour) > 0 else np.nan
    )

    print(f"→ n_festooning={n_above}, mean_top100_score={mean_top100_score:.4f}")
    results.append({
        "hour": hour_idx, "frame_start": h_start,
        "n_above_threshold": n_above, "mean_top100_score": mean_top100_score,
    })

    # ==================================================================
    # Repeat the same metrics separately for every age group
    # ==================================================================
    for age in age_groups:
        pbc_age = pbc[pbc["age"] == age]
        tpc_age = tpc[tpc["age"] == age]

        if pbc_age.empty:
            results_by_age[age].append({
                "hour": hour_idx,
                "n_above_threshold": 0, "mean_top100_score": np.nan,
            })
            continue

        best_age   = pbc_age.groupby("uid")["festoon_score"].min().reset_index()
        n_above_age = int((best_age["festoon_score"] <= threshold_score).sum())

        chunk_app_age = (
            tpc_age.groupby("uid").agg(chunks_in_top=("chunk", "count"))
                   .reset_index().sort_values("chunks_in_top", ascending=False)
        )
        top100_age     = chunk_app_age.head(TOP_FESTOON)["uid"].values
        mean_score_age = (
            pbc_age[pbc_age["uid"].isin(top100_age)]["festoon_score"].mean()
            if len(top100_age) > 0 else np.nan
        )

        results_by_age[age].append({
            "hour":               hour_idx,
            "n_above_threshold":  n_above_age,
            "mean_top100_score":  mean_score_age,
        })

# ======================================================================
# Convert collected results to dataframes
# ======================================================================
results_df    = pd.DataFrame(results)
dfs_by_age    = {a: pd.DataFrame(results_by_age[a]) for a in age_groups}
print("\nDone.")
print(results_df)


In [ ]:
## Plotting festoon activity across the full day

# ======================================================================
# General plot configuration
# ======================================================================
hours     = results_df["hour"]
n_ages    = len(age_groups)
n_cols    = 3
n_rows    = int(np.ceil(n_ages / n_cols))
color_map = {a: plt.cm.tab10(i % 10) for i, a in enumerate(age_groups)}

# Locate the original reference window in the hourly results so that it
# can be shaded consistently on the plots.
ref_hour_start = next(i for i, h in enumerate(hour_starts) if h >= FRAME_START)
ref_hour_end   = next((i for i, h in enumerate(hour_starts) if h >= FRAME_END),
                       len(hour_starts)) - 1

def shade_ref(ax):
    ax.axvspan(ref_hour_start - 0.5, ref_hour_end + 0.5,
               alpha=0.12, color="orange", label="Reference window")

# ======================================================================
# 4a. Overall hourly festoon activity
# ======================================================================
# Panel 1: number of bees meeting the threshold.
# Panel 2: mean score of the top persistent candidates.
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax1.bar(hours, results_df["n_above_threshold"],
        color="steelblue", edgecolor="black", linewidth=0.6)
ax1.axhline(TOP_FESTOON, color="red", linestyle="--", linewidth=1,
            label=f"Reference top-{TOP_FESTOON} count")
shade_ref(ax1)
ax1.set_ylabel("Unique bees above threshold")
ax1.set_title(f"Festooning bees per hour  (threshold = {threshold_score:.4f})")
ax1.legend(); ax1.grid(axis="y", alpha=0.3)

ax2.plot(hours, results_df["mean_top100_score"],
         marker="o", color="darkorange", linewidth=2, markersize=5)
ax2.axhline(threshold_score, color="red", linestyle="--", linewidth=1,
            label=f"Threshold ({threshold_score:.4f})")
shade_ref(ax2)
ax2.set_xlabel("Hour of day")
ax2.set_ylabel("Mean festoon score (lower = better)")
ax2.set_title(f"Mean festoon score of top-{TOP_FESTOON} bees per hour")
ax2.set_xticks(hours)
ax2.set_xticklabels([str(h) for h in hours], fontsize=8)
ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle(f"Day {DAY} — Festoon activity across the day", fontsize=13)
plt.tight_layout()
plt.savefig(f"festoon_per_hour_day{DAY}.png", dpi=150, bbox_inches="tight")
plt.show()

# ======================================================================
# 4b. Per-age hourly count of festooning bees
# ======================================================================
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows), sharex=True)
axes = axes.flatten()
fig.suptitle(f"Day {DAY} — Festooning bees per hour by age  "
             f"(threshold = {threshold_score:.4f})", fontsize=13)

for idx, age in enumerate(age_groups):
    ax     = axes[idx]
    df_age = dfs_by_age[age]
    ax.bar(df_age["hour"], df_age["n_above_threshold"],
           color=color_map[age], edgecolor="black", linewidth=0.5, alpha=0.85)
    shade_ref(ax)
    ax.set_title(f"Age {int(age)} days")
    ax.set_ylabel("Bees above threshold")
    ax.set_xticks(hours)
    ax.set_xticklabels([str(h) for h in hours], fontsize=6, rotation=45)
    ax.grid(axis="y", alpha=0.3)

for j in range(idx + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig(f"festoon_per_hour_by_age_count_day{DAY}.png", dpi=150, bbox_inches="tight")
plt.show()

# ======================================================================
# 4c. Per-age hourly mean festoon score
# ======================================================================
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows), sharex=True)
axes = axes.flatten()
fig.suptitle(f"Day {DAY} — Mean festoon score per hour by age", fontsize=13)

for idx, age in enumerate(age_groups):
    ax     = axes[idx]
    df_age = dfs_by_age[age]
    ax.plot(df_age["hour"], df_age["mean_top100_score"],
            marker="o", color=color_map[age], linewidth=2, markersize=4)
    ax.axhline(threshold_score, color="red", linestyle="--", linewidth=1,
               label=f"Threshold ({threshold_score:.4f})")
    shade_ref(ax)
    ax.set_title(f"Age {int(age)} days")
    ax.set_ylabel("Mean festoon score")
    ax.set_xticks(hours)
    ax.set_xticklabels([str(h) for h in hours], fontsize=6, rotation=45)
    ax.legend(fontsize=6); ax.grid(alpha=0.3)

for j in range(idx + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig(f"festoon_per_hour_by_age_score_day{DAY}.png", dpi=150, bbox_inches="tight")
plt.show()


# ANIMATION

In [ ]:
## Creating the festoon animation

import os
import pickle
import subprocess
import numpy as np
import sys

# ======================================================================
# Define the animation frame range and input data
# ======================================================================
START_FRAME = FRAME_START
END_FRAME   = FRAME_END

# df_box already contains the data restricted to the analysis box and
# reference frame range.
df_range = df_box.copy()

target_minutes = (END_FRAME - START_FRAME) / (3 * 60)   # at 3 FPS

# ======================================================================
# Animation configuration
# ======================================================================
N_WORKERS   = 14
STRIDE      = 1
FRAME_LIMIT = None
OUTPUT_DIR  = "chunks_festoon"
SLICES_DIR  = "slices_festoon"
FPS         = 3.0

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SLICES_DIR, exist_ok=True)

# ======================================================================
# Split the animation frames across worker processes
# ======================================================================
grouped    = df_range.groupby("framenum")
all_frames = sorted(grouped.groups.keys())[::STRIDE]
if FRAME_LIMIT:
    all_frames = all_frames[:FRAME_LIMIT]

frame_chunks = [c.tolist() for c in np.array_split(all_frames, N_WORKERS)]
print(f"Total frames: {len(all_frames)} | Workers: {N_WORKERS}")

# ======================================================================
# Build lookups for festoon candidates and scores
# ======================================================================
# chunk_uid_map stores the top candidate UIDs in score order for each chunk.
chunk_uid_map = {}   # chunk_idx -> list of uids
for chunk_idx, grp in top_per_chunk.groupby("chunk"):
    chunk_uid_map[chunk_idx] = (
        grp.sort_values("festoon_score")["uid"].head(TOP_N).tolist()
    )

# score_lookup allows the worker to retrieve a candidate's score from
# its chunk and UID.
score_lookup = {}
for _, row in top_per_chunk.iterrows():
    score_lookup[(int(row["chunk"]), int(row["uid"]))] = row["festoon_score"]

# ======================================================================
# Define age colours
# ======================================================================
age_list = sorted(df["age"].dropna().unique())

RED_SAFE_COLORS = [
    "#00FFFF",  # cyan
    "#FFFFFF",  # white
    "#39FF14",  # neon lime
    "#FF00FF",  # magenta
    "#FFD700",  # gold
    "#00BFFF",  # deep sky blue
    "#FF69B4",  # hot pink
]
age_colors = {a: RED_SAFE_COLORS[i % len(RED_SAFE_COLORS)] for i, a in enumerate(age_list)}

# ======================================================================
# Save each frame slice for its worker
# ======================================================================
slice_paths = []
for i, frame_chunk in enumerate(frame_chunks):
    df_slice   = df_range[df_range["framenum"].isin(frame_chunk)]
    slice_path = os.path.join(SLICES_DIR, f"slice_{i:04d}.pkl")

    with open(slice_path, "wb") as f:
        pickle.dump({
            "df_slice"      : df_slice,
            "frame_indices" : frame_chunk,
            "age_colors"    : age_colors,
            "age_list"      : age_list,
            "xpixels"       : bd.xpixels,
            "DAY"           : DAY,
            "comb"          : comb,
            "FPS"            : FPS,
            "chunk_uid_map" : chunk_uid_map,
            "score_lookup"  : score_lookup,
            "FRAME_START"   : FRAME_START,
            "CHUNK_SIZE"    : CHUNK_SIZE,
            "BOX_X"         : BOX_X,
            "BOX_Y"         : BOX_Y,
        }, f)

    slice_paths.append(slice_path)
    print(f"  Slice {i}: {len(df_slice)} rows, {len(frame_chunk)} frames → {slice_path}")

# ======================================================================
# Launch all animation workers in parallel
# ======================================================================
procs = []
for i, slice_path in enumerate(slice_paths):
    cmd = [sys.executable, "worker_festoon.py", str(i), slice_path, OUTPUT_DIR]
    p   = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    procs.append((i, p))
    print(f"Launched worker {i} (pid {p.pid})")

# ======================================================================
# Stream worker output while the processes are running
# ======================================================================
import threading

def stream_output(chunk_id, proc):
    for line in proc.stdout:
        print(line, end="", flush=True)

threads = [threading.Thread(target=stream_output, args=(i, p)) for i, p in procs]
for t in threads: t.start()
for t in threads: t.join()

for i, p in procs:
    p.wait()
    print(f"Worker {i} exited with code {p.returncode}")

# ======================================================================
# Merge the worker-generated video chunks
# ======================================================================
chunk_paths = sorted([
    os.path.join(OUTPUT_DIR, f)
    for f in os.listdir(OUTPUT_DIR) if f.endswith(".mp4")
])

list_file = "chunk_list_festoon.txt"
with open(list_file, "w") as f:
    for path in chunk_paths:
        f.write(f"file '{os.path.abspath(path)}'\n")

subprocess.run([
    "ffmpeg", "-y", "-f", "concat", "-safe", "0",
    "-i", list_file, "-c", "copy",
    f"festoon_animation_{START_FRAME}_{END_FRAME}.mp4"
], check=True)

os.remove(list_file)
print(f"Final video → festoon_animation_{START_FRAME}_{END_FRAME}_Day48.mp4")


In [ ]:
## Cleaning the festoon animation output directories

import os, shutil

# Remove incomplete/intermediate worker outputs and slice pickles.
# Run this before rerunning the animation if old chunks/slices should not
# be reused.
for d in ["chunks_festoon", "slices_festoon"]:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"Removed {d}/")
